# 1D Breakout Model Extraction and Comparison

This notebook extracts a continuous portion of a real steady 1D HEC-RAS reach into an independent project, validates the retained model content, runs both projects, and compares the retained-section results. Plan-view figures show the source geometry, selected interval, retained breakout elements, and a conceptual network-conflation handoff. The example uses the official **Balde Eagle Creek** project and keeps an intervening bridge structure.

The workflow demonstrates mechanics, not a universal acceptance standard. Choose station limits, boundary methods, and comparison tolerances using the governing project requirements and applicable HEC-RAS guidance.

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from shapely.ops import nearest_points

import ras_commander
from ras_commander import (
    GeomBridge,
    GeomParser,
    HdfXsec,
    RasBreakout1D,
    RasCmdr,
    RasExamples,
    RasPrj,
    init_ras_project,
)

print(f"ras-commander: {ras_commander.__version__}")
print(f"Loaded from: {ras_commander.__file__}")

## Configure the real example

The selected interval contains 11 natural cross sections and the bridge near river station 103245. The source plan is run first because an internal downstream cut needs source-plan water-surface results to create known-WSE boundary conditions for each steady profile.

HEC-RAS on Windows must see the working folder through a local or mapped-drive path; it cannot compute a project referenced through a UNC path.

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "ras_commander").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
WORK_ROOT = REPO_ROOT / "working" / "235_1d_breakout_model"
RUN_ROOT = WORK_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
SOURCE_PLAN = "02"
RAS_VERSION = "7.0"
RIVER = "Bald Eagle"
REACH = "Loc Hav"
UPSTREAM_STATION = 106466.0
DOWNSTREAM_STATION = 98206.87

if str(RUN_ROOT).startswith("\\"):
    raise RuntimeError("Use a local or mapped-drive working path for HEC-RAS execution.")

RUN_ROOT.mkdir(parents=True, exist_ok=False)
print(f"Run workspace: {RUN_ROOT}")

## 1. Extract and initialize the source project

`RasExamples.extract_project()` creates a disposable working copy. The packaged source remains unchanged.

In [ ]:
source_path = RasExamples.extract_project(
    "Balde Eagle Creek",
    output_path=RUN_ROOT / "source",
)
source_ras = RasPrj()
init_ras_project(source_path, RAS_VERSION, ras_object=source_ras)

source_plan = source_ras.plan_df.loc[
    source_ras.plan_df["plan_number"].astype(str).str.zfill(2) == SOURCE_PLAN
].iloc[0]
display(
    source_ras.plan_df[[
        "plan_number", "Plan Title", "flow_type", "geometry_type",
        "Geom Path", "Flow Path",
    ]]
)

## 2. Run the source steady plan

The resulting plan HDF supplies profile-specific known water-surface elevations at the new downstream limit and provides the baseline results for comparison.

In [ ]:
source_compute = RasCmdr.compute_plan(
    SOURCE_PLAN,
    ras_object=source_ras,
    clear_geompre=True,
    force_rerun=True,
    num_cores=1,
    verify=True,
)
assert source_compute, "Source steady plan did not complete successfully"

source_plan_hdf = Path(f"{source_plan['full_path']}.hdf")
source_geometry = Path(source_plan["Geom Path"])
assert source_plan_hdf.is_file()
print(f"Source results: {source_plan_hdf}")

## 3. Resolve the retained reach slice

Selection is separate from writing. Other workflows can replace this call with `select_by_polygon()`, `select_by_network_segment()`, or `select_by_cross_sections()` and pass the resulting selection to the same extractor.

In [ ]:
selection = RasBreakout1D.select_by_stations(
    source_geometry,
    river=RIVER,
    reach=REACH,
    upstream_station=UPSTREAM_STATION,
    downstream_station=DOWNSTREAM_STATION,
)

display(pd.DataFrame({
    "river": [selection.river],
    "reach": [selection.reach],
    "upstream_station": [selection.upstream_station],
    "downstream_station": [selection.downstream_station],
    "retained_xs_count": [len(selection.stations)],
    "selector": [selection.selector],
}))
selection.stations

### Source geometry and selected interval

The left panel places the selection within the complete 138,000-foot source reach. The detail panel shows every retained cross-section cut line, both new model limits, the river centerline, and the bridge that must remain between those limits. The shaded model domain comes from the HEC-RAS cross-section interpolation surface rather than an invented buffer.

In [ ]:
source_geometry_hdf = Path(f"{source_geometry}.hdf")
source_xs = GeomParser.get_xs_cut_lines(source_geometry)
source_centerlines = GeomParser.get_river_centerlines(source_geometry)
source_surface = HdfXsec.get_xs_interpolation_surface(source_geometry_hdf)

source_reach_xs = source_xs.loc[
    (source_xs["river"] == RIVER) & (source_xs["reach"] == REACH)
].copy()
source_reach_xs["station_num"] = pd.to_numeric(
    source_reach_xs["station"], errors="coerce"
)
selected_station_numbers = np.asarray([float(value) for value in selection.stations])
source_reach_xs["selected"] = source_reach_xs["station_num"].apply(
    lambda value: bool(np.isclose(value, selected_station_numbers).any())
)
selected_source_xs = source_reach_xs.loc[source_reach_xs["selected"]].copy()
excluded_source_xs = source_reach_xs.loc[~source_reach_xs["selected"]].copy()

selected_xs_ids = selected_source_xs.index.to_numpy(dtype=int)
source_selected_surface = source_surface.loc[
    (source_surface["us_xs_id"] >= selected_xs_ids.min())
    & (source_surface["ds_xs_id"] <= selected_xs_ids.max())
].copy()
assert not source_selected_surface.empty

reach_centerline = source_centerlines.loc[
    (source_centerlines["river"] == RIVER)
    & (source_centerlines["reach"] == REACH),
    "geometry",
].iloc[0]


def channel_point(cut_line, centerline):
    return nearest_points(cut_line, centerline)[1]


def station_channel_point(xs_frame, station, centerline):
    row = xs_frame.iloc[(xs_frame["station_num"] - float(station)).abs().argmin()]
    return channel_point(row.geometry, centerline)


def structure_channel_point(xs_frame, station, centerline):
    upstream = xs_frame.loc[xs_frame["station_num"] > station].nsmallest(1, "station_num").iloc[0]
    downstream = xs_frame.loc[xs_frame["station_num"] < station].nlargest(1, "station_num").iloc[0]
    upstream_measure = centerline.project(channel_point(upstream.geometry, centerline))
    downstream_measure = centerline.project(channel_point(downstream.geometry, centerline))
    fraction = (upstream.station_num - station) / (upstream.station_num - downstream.station_num)
    return centerline.interpolate(
        upstream_measure + fraction * (downstream_measure - upstream_measure)
    )


source_bridges = GeomBridge.get_bridges(source_geometry)
selected_bridges = source_bridges.loc[
    (source_bridges["River"] == RIVER)
    & (source_bridges["Reach"] == REACH)
    & pd.to_numeric(source_bridges["RS"], errors="coerce").between(
        DOWNSTREAM_STATION, UPSTREAM_STATION
    )
].copy()
bridge_points = [
    (float(row.RS), structure_channel_point(source_reach_xs, float(row.RS), reach_centerline))
    for row in selected_bridges.itertuples(index=False)
]
upstream_point = station_channel_point(source_reach_xs, UPSTREAM_STATION, reach_centerline)
downstream_point = station_channel_point(source_reach_xs, DOWNSTREAM_STATION, reach_centerline)

domain_color = "#E5E7EB"
selection_color = "#F6C85F"
centerline_color = "#374151"
retained_color = "#0072B2"
bridge_color = "#D55E00"
upstream_color = "#009E73"
downstream_color = "#CC79A7"

fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
source_surface.plot(ax=axes[0], color=domain_color, edgecolor="none", alpha=0.75)
source_selected_surface.plot(
    ax=axes[0], color=selection_color, edgecolor=bridge_color, linewidth=1.0, alpha=0.85
)
source_centerlines.plot(ax=axes[0], color=centerline_color, linewidth=1.2)
excluded_source_xs.plot(ax=axes[0], color="#9CA3AF", linewidth=0.35, alpha=0.55)
selected_source_xs.plot(ax=axes[0], color=retained_color, linewidth=1.6)
axes[0].set_title("Source-model context")

source_selected_surface.plot(
    ax=axes[1], color=selection_color, edgecolor=bridge_color, linewidth=1.0, alpha=0.45
)
source_centerlines.plot(ax=axes[1], color=centerline_color, linewidth=2.0)
selected_source_xs.plot(ax=axes[1], color=retained_color, linewidth=1.7)
label_stations = [float(selection.stations[3]), float(selection.stations[7])]
for station in label_stations:
    row = selected_source_xs.iloc[(selected_source_xs["station_num"] - station).abs().argmin()]
    point = channel_point(row.geometry, reach_centerline)
    axes[1].annotate(
        f"RS {row.station}",
        (point.x, point.y),
        xytext=(6, 5),
        textcoords="offset points",
        fontsize=8,
    )
for station, point in bridge_points:
    axes[1].scatter(point.x, point.y, marker="*", s=180, color=bridge_color, zorder=8)
    axes[1].annotate(
        f"Bridge RS {station:g}", (point.x, point.y), xytext=(8, 10),
        textcoords="offset points", color=bridge_color, fontsize=9, fontweight="bold",
    )
axes[1].scatter(
    upstream_point.x, upstream_point.y, marker="s", s=65, color=upstream_color, zorder=9
)
axes[1].scatter(
    downstream_point.x, downstream_point.y, marker="s", s=65, color=downstream_color, zorder=9
)
axes[1].annotate("New upstream limit", (upstream_point.x, upstream_point.y), xytext=(8, -18), textcoords="offset points", fontsize=9)
axes[1].annotate("New downstream limit", (downstream_point.x, downstream_point.y), xytext=(8, -18), textcoords="offset points", fontsize=9)
xmin, ymin, xmax, ymax = source_selected_surface.total_bounds
padding = 0.06 * max(xmax - xmin, ymax - ymin)
axes[1].set_xlim(xmin - padding, xmax + padding)
axes[1].set_ylim(ymin - padding, ymax + padding)
axes[1].set_title("Selected breakout interval")

legend_handles = [
    Patch(facecolor=domain_color, label="Source interpolation surface"),
    Patch(facecolor=selection_color, edgecolor=bridge_color, label="Selected model domain"),
    Line2D([0], [0], color=retained_color, linewidth=1.8, label="Retained cross section"),
    Line2D([0], [0], color=centerline_color, linewidth=2.0, label="River centerline"),
    Line2D([0], [0], marker="*", color=bridge_color, linestyle="none", markersize=12, label="Retained bridge"),
]
axes[0].legend(handles=legend_handles, loc="best", fontsize=8)
for ax in axes:
    ax.set_aspect("equal")
    ax.set_axis_off()
fig.suptitle("Where the 1D breakout sits inside the source model", fontsize=14)
plt.show()

### How network conflation hands limits to the breakout

The figure below is deliberately schematic; it does **not** claim that the displayed lines are downloaded NextGen flowpaths for Bald Eagle Creek. It separates two decisions: the model polygon determines which directed network edges are inside the hydraulic domain, while cross-section intersections and network connectivity determine the extraction limits supplied to `RasBreakout1D`. Alignment measures remain QA diagnostics rather than competing default match scores.

In [ ]:
extent_start, extent_end = 1.0, 9.2
network_edges = [
    ("edge 101", -0.8, 2.4),
    ("edge 102", 2.4, 4.8),
    ("edge 103", 4.8, 6.7),
    ("edge 104", 6.7, 10.7),
]
xs_locations = [(1.7, "XS A"), (3.2, "XS B"), (4.4, "XS C"), (7.5, "XS D"), (8.8, "XS E")]
edge_colors = ["#E69F00", "#0072B2", "#999999", "#CC79A7"]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)
axes[0].add_patch(Rectangle(
    (extent_start, -1.1), extent_end - extent_start, 2.2,
    facecolor=selection_color, edgecolor=bridge_color, alpha=0.25, linewidth=1.5,
))
axes[0].text((extent_start + extent_end) / 2, 0.96, "RAS model extent polygon", ha="center", fontsize=11)
for (edge_id, start, end), color in zip(network_edges, edge_colors):
    inside_length = max(0.0, min(end, extent_end) - max(start, extent_start))
    inside_fraction = inside_length / (end - start)
    status = "fully inside" if np.isclose(inside_fraction, 1.0) else "boundary edge"
    if edge_id == "edge 103":
        status = "inside; no XS intersection → eclipsed"
    axes[0].plot([start, end], [0, 0], color=color, linewidth=7, solid_capstyle="butt")
    axes[0].scatter([start, end], [0, 0], color=color, s=35, zorder=4)
    axes[0].text(
        (start + end) / 2, -0.32,
        f"{edge_id}\n{inside_fraction:.0%} inside\n{status}",
        ha="center", va="top", fontsize=9,
    )
for x, label in xs_locations:
    axes[0].plot([x, x], [-0.68, 0.68], color=retained_color, linewidth=1.8)
    axes[0].text(x, 0.55, label, ha="center", fontsize=9)
axes[0].annotate(
    "directed network flow", xy=(9.8, 1.35), xytext=(1.2, 1.35),
    arrowprops={"arrowstyle": "->", "linewidth": 1.8}, va="center", fontsize=10,
)
axes[0].set_xlim(-1.1, 11.0)
axes[0].set_ylim(-1.65, 1.7)
axes[0].axis("off")
axes[0].set_title("1 — Classify every network edge by line length inside the model polygon")

steps = [
    (1.0, "Model polygon\nclassifies edges"),
    (4.0, "Connectivity preserves\nthe in-domain path"),
    (7.0, "XS intersections set\nupstream/downstream limits"),
    (10.0, "RasBreakout1D writes,\nruns, and compares"),
]
for index, (x, label) in enumerate(steps):
    axes[1].scatter(x, 0, s=650, color=[selection_color, retained_color, upstream_color, bridge_color][index], alpha=0.9, zorder=3)
    axes[1].text(x, 0, str(index + 1), ha="center", va="center", fontsize=12, fontweight="bold", color="white")
    axes[1].text(x, -0.46, label, ha="center", va="top", fontsize=10)
    if index < len(steps) - 1:
        axes[1].annotate("", xy=(steps[index + 1][0] - 0.45, 0), xytext=(x + 0.45, 0), arrowprops={"arrowstyle": "->", "linewidth": 1.8})
axes[1].set_xlim(0, 11)
axes[1].set_ylim(-1.25, 0.75)
axes[1].axis("off")
axes[1].set_title("2 — Convert spatial coverage into an auditable breakout selection")
fig.suptitle("Extent-first conflation and breakout handoff (conceptual)", fontsize=14)
plt.show()

## 4. Write and structurally validate the breakout

The extractor copies complete retained cross-section and intervening structure blocks, carries the applicable flow changes, creates an independent `p01/g01/f01` project, zeroes the new downstream reach lengths, and assigns known-WSE downstream boundaries from the source results.

In [ ]:
breakout = RasBreakout1D.extract_selection(
    source_ras,
    RUN_ROOT / "breakout",
    selection,
    plan_number=SOURCE_PLAN,
    destination_name="Breakout",
    source_plan_hdf=source_plan_hdf,
    boundary_mode="auto",
)

display(breakout.validation.checks_df)
assert breakout.validation.is_valid
print(f"Boundary provenance: {breakout.boundary_provenance}")
print(f"Independent project: {breakout.project_file}")

## 5. Compare retained geometry before execution

This is an exact payload comparison for each retained cross section. The table attributes also report whether all intervening structure blocks match.

In [ ]:
geometry_comparison = RasBreakout1D.compare_geometry(
    source_geometry,
    breakout.geometry_file,
    breakout.selection,
)
display(geometry_comparison)
print(geometry_comparison.attrs)

assert geometry_comparison["content_equal"].all()
assert geometry_comparison.attrs["structure_blocks_equal"] is True

## 6. Run the independent breakout

Execution remains explicit. `RasBreakout1D.run()` delegates to `RasCmdr.compute_plan()` using the destination `RasPrj`.

In [ ]:
breakout_compute = RasBreakout1D.run(
    breakout,
    verify=True,
    force_rerun=True,
    num_cores=1,
)
assert breakout_compute, "Breakout plan did not complete successfully"

breakout_plan_hdf = Path(f"{breakout.plan_file}.hdf")
assert breakout_plan_hdf.is_file()
print(f"Breakout results: {breakout_plan_hdf}")

### Source interval versus the computed breakout geometry

These panels use the source and destination geometry files independently. The left panel retains neighboring source cross sections as muted context; the right panel contains only the breakout elements HEC-RAS actually computed. Matching positions are necessary but not sufficient, which is why the exact block comparison above remains part of the audit.

In [ ]:
breakout_geometry_hdf = Path(f"{breakout.geometry_file}.hdf")
breakout_xs = GeomParser.get_xs_cut_lines(breakout.geometry_file)
breakout_centerlines = GeomParser.get_river_centerlines(breakout.geometry_file)
breakout_surface = HdfXsec.get_xs_interpolation_surface(breakout_geometry_hdf)
breakout_bridges = GeomBridge.get_bridges(breakout.geometry_file)
breakout_xs["station_num"] = pd.to_numeric(breakout_xs["station"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True, sharey=True, constrained_layout=True)
source_selected_surface.plot(
    ax=axes[0], color=selection_color, edgecolor=bridge_color, linewidth=1.0, alpha=0.38
)
excluded_source_xs.plot(ax=axes[0], color="#9CA3AF", linewidth=0.7, alpha=0.45)
selected_source_xs.plot(ax=axes[0], color=retained_color, linewidth=1.8)
source_centerlines.plot(ax=axes[0], color=centerline_color, linewidth=2.0)
axes[0].set_title(
    f"Source selection: {len(selected_source_xs)} XS, {len(selected_bridges)} bridge"
)

breakout_surface.plot(
    ax=axes[1], color="#B9E3C6", edgecolor=upstream_color, linewidth=1.0, alpha=0.48
)
breakout_xs.plot(ax=axes[1], color=retained_color, linewidth=1.8)
breakout_centerlines.plot(ax=axes[1], color=centerline_color, linewidth=2.0)
axes[1].set_title(
    f"Independent breakout: {len(breakout_xs)} XS, {len(breakout_bridges)} bridge"
)

for ax in axes:
    for station, point in bridge_points:
        ax.scatter(point.x, point.y, marker="*", s=180, color=bridge_color, zorder=8)
        ax.annotate(f"Bridge RS {station:g}", (point.x, point.y), xytext=(7, 8), textcoords="offset points", fontsize=9, color=bridge_color)
    ax.scatter(upstream_point.x, upstream_point.y, marker="s", s=65, color=upstream_color, zorder=9)
    ax.scatter(downstream_point.x, downstream_point.y, marker="s", s=65, color=downstream_color, zorder=9)
    ax.annotate("US limit", (upstream_point.x, upstream_point.y), xytext=(6, -16), textcoords="offset points", fontsize=9)
    ax.annotate("DS limit", (downstream_point.x, downstream_point.y), xytext=(6, -16), textcoords="offset points", fontsize=9)
    ax.set_xlim(xmin - padding, xmax + padding)
    ax.set_ylim(ymin - padding, ymax + padding)
    ax.set_aspect("equal")
    ax.set_axis_off()
axes[1].legend(
    handles=[
        Patch(facecolor="#B9E3C6", edgecolor=upstream_color, label="Computed breakout surface"),
        Line2D([0], [0], color=retained_color, linewidth=1.8, label="Retained cross section"),
        Line2D([0], [0], marker="*", color=bridge_color, linestyle="none", markersize=12, label="Retained bridge"),
    ],
    loc="best", fontsize=8,
)
fig.suptitle("Spatial audit of retained breakout elements", fontsize=14)
plt.show()

## 7. Compare retained-section hydraulic results

The comparison joins by river, reach, station, and profile and adds a delta for every numeric result available in both HDF files. All retained station/profile keys should join as `both`.

The new downstream cross section intentionally has zero reach lengths, so `channel_length_delta` is expected and is not a hydraulic mismatch. For this deterministic example, focus on flow, WSE, velocity, depth, top width, area, and slope deltas. Any project-specific acceptance threshold should be selected independently of this demonstration.

In [ ]:
results_comparison = RasBreakout1D.compare_results(
    source_plan_hdf,
    breakout_plan_hdf,
    breakout.selection,
)
assert results_comparison["_merge"].eq("both").all()

delta_columns = [
    column for column in results_comparison.columns
    if column.endswith("_delta") and column != "channel_length_delta"
]
delta_summary = (
    results_comparison[delta_columns]
    .abs()
    .agg(["count", "mean", "max"])
    .T
    .sort_index()
)
display(delta_summary)

preview_columns = [
    "river", "reach", "node_id", "profile",
    "flow_source", "flow_destination", "flow_delta",
    "wsel_source", "wsel_destination", "wsel_delta", "_merge",
]
display(results_comparison[preview_columns].head(16))

### Longitudinal hydraulic comparison

The highest-flow profile provides the clearest visual check. The upper panel overlays source and breakout water surfaces on the minimum surveyed elevation at each source cross section; the lower panel expands the very small water-surface differences into inches.

In [ ]:
profile_flows = results_comparison.groupby("profile", observed=True)["flow_source"].max()
comparison_profile = profile_flows.idxmax()
profile_plot = results_comparison.loc[
    results_comparison["profile"] == comparison_profile
].copy()
profile_plot["station_num"] = pd.to_numeric(profile_plot["node_id"], errors="coerce")
profile_plot = profile_plot.sort_values("station_num", ascending=False)
source_profile_xs = HdfXsec.get_cross_sections(source_geometry_hdf)
source_profile_xs = source_profile_xs.loc[
    (source_profile_xs["River"] == RIVER) & (source_profile_xs["Reach"] == REACH)
].copy()
source_profile_xs["station_num"] = pd.to_numeric(source_profile_xs["RS"], errors="coerce")
source_profile_xs["channel_invert"] = source_profile_xs["station_elevation"].apply(
    lambda points: min(float(point[1]) for point in points)
)
profile_plot = profile_plot.merge(
    source_profile_xs[["station_num", "channel_invert"]], on="station_num", how="left"
)

fig, axes = plt.subplots(
    2, 1, figsize=(13, 7), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]}, constrained_layout=True,
)
axes[0].plot(
    profile_plot["station_num"], profile_plot["channel_invert"],
    color=centerline_color, linewidth=1.8, marker=".", label="Source channel invert",
)
axes[0].fill_between(
    profile_plot["station_num"], profile_plot["channel_invert"],
    profile_plot["channel_invert"].min() - 2.0, color=domain_color, alpha=0.8,
)
axes[0].plot(
    profile_plot["station_num"], profile_plot["wsel_source"],
    color=retained_color, linewidth=2.4, marker="o", markersize=4, label="Source WSE",
)
axes[0].plot(
    profile_plot["station_num"], profile_plot["wsel_destination"],
    color=bridge_color, linewidth=1.8, linestyle="--", marker="s", markersize=3.5,
    label="Breakout WSE",
)
axes[0].set_ylabel("Elevation (ft)")
axes[0].set_title(
    f"{comparison_profile} profile — {profile_flows.loc[comparison_profile]:,.0f} cfs"
)
axes[0].ticklabel_format(style="plain", axis="y", useOffset=False)
axes[0].grid(axis="y", alpha=0.25)
axes[0].legend(loc="best")

wse_delta_inches = profile_plot["wsel_delta"] * 12.0
axes[1].axhline(0.0, color=centerline_color, linewidth=1.0)
axes[1].plot(
    profile_plot["station_num"], wse_delta_inches,
    color=upstream_color, linewidth=1.8, marker="o", markersize=4,
)
axes[1].fill_between(
    profile_plot["station_num"], 0.0, wse_delta_inches, color=upstream_color, alpha=0.18
)
axes[1].set_ylabel("WSE delta (in)")
axes[1].set_xlabel("River station (ft; flow direction →)")
axes[1].grid(axis="y", alpha=0.25)
axes[1].invert_xaxis()
max_delta_inches = float(wse_delta_inches.abs().max())
delta_limit = max(max_delta_inches * 1.25, 0.001)
axes[1].set_ylim(-delta_limit, delta_limit)
axes[1].text(
    0.01, 0.93, f"Maximum |WSE delta| = {max_delta_inches:.6f} in",
    transform=axes[1].transAxes, va="top", fontsize=9,
)
fig.suptitle("Source and breakout hydraulic equivalence", fontsize=14)
plt.show()

## Review products

The workflow leaves two independently openable HEC-RAS projects under `RUN_ROOT`, tabular audit products in memory, and four visual review products rendered above:

- `breakout.validation.checks_df` — structural and relationship checks;
- `geometry_comparison` — exact retained geometry and structure-block agreement;
- `results_comparison` and `delta_summary` — retained-section/profile hydraulic differences;
- plan-view source/selection and source/breakout figures — spatial retention evidence;
- the conflation schematic and longitudinal comparison — selection logic and hydraulic equivalence.

For multi-reach, junction, lateral-structure, or unsteady extraction, stop at selection and use a workflow that explicitly supports those model elements; the current `RasBreakout1D` contract intentionally fails closed outside its one-reach steady scope.